# Training behaviour

Four knobs, same customer table, same split as `00_mlp_churn.ipynb`.

- Learning rate is the step size. This table uses SGD, so the rate you set is the step. The morning MLP used Adam, which rescales the step. A rate that is too large can score well on the train rows and worse on the validation rows.
- Scaling puts the numeric columns on a similar range before the step. Spend in baht is much larger than a share between 0 and 1.
- Early stopping watches a slice of the train rows and stops when that slice stops improving.
- A learning curve plots the score against how many train rows the model saw.

The iteration cap is part of the result. A run that hits the cap has not finished moving the weights.


In [ ]:
import os
import sys
from pathlib import Path

os.environ["MLFLOW_ALLOW_FILE_STORE"] = "true"
os.environ["MLFLOW_DISABLE_AGENT_HINT"] = "1"

here = Path.cwd().resolve()
root = None
for candidate in [here, *here.parents]:
    for folder in (candidate, candidate / "04-deep-learning"):
        marker = folder / "lib" / "customers.py"
        if marker.is_file() and (folder / "app.py").is_file():
            root = folder
            break
    if root is not None:
        break
if root is None:
    raise FileNotFoundError("Start Jupyter from the repo folder or from 04-deep-learning.")
lib = root / "lib"
if str(lib) not in sys.path:
    sys.path.insert(0, str(lib))
ROOT = root
print("module", ROOT)


In [ ]:
from customers import load_customers, split_frame
from training import curve_table, learning_rate_table, scaling_table, stopping_table

customers = load_customers()
train, validation, test = split_frame(customers)
print("train", len(train), "validation", len(validation), "test left unused", len(test))


In [ ]:
rates = learning_rate_table(train, validation)
print(rates.to_string(index=False, float_format=lambda value: f"{value:.3f}"))


In [ ]:
scaled = scaling_table(train, validation)
print(scaled.to_string(index=False, float_format=lambda value: f"{value:.3f}"))


## Early stopping

Exercise. Read the table. `gap` is the train ROC-AUC minus the validation ROC-AUC for the row `no early stopping`. Use the printed numbers, three decimals.


In [ ]:
stop = stopping_table(train, validation)
print(stop.to_string(index=False, float_format=lambda value: f"{value:.3f}"))

gap = None  # Exercise. Train ROC-AUC minus validation ROC-AUC for "no early stopping".
if gap is None:
    print("Fill gap, then run this cell again.")
else:
    print("gap", round(gap, 3))


## Learning curve

`learning_curve` uses only the train rows. The cv score is a 3-fold score inside those rows, not the validation split above. The largest row count is about two thirds of the train rows, because one fold is held out.


In [ ]:
curve = curve_table(train)
print(curve.to_string(index=False, float_format=lambda value: f"{value:.3f}"))


## Debrief

Exercise. One sentence on the train score versus the validation score. Which setting opened the widest gap?


In [ ]:
debrief = ""  # Exercise. One sentence.
print(debrief or "Write one sentence, then run this cell again.")
